# RAG Debugging Lab: Diagnosing Retrieval Failures with Arize Phoenix

**This is a hands-on lab, not a read-only cookbook.** You'll walk through one fully-worked
failure case (Exercise 1), then diagnose and fix a second one mostly on your own (Exercise 2),
using Phoenix's tracing and eval tooling both times.

**What makes this lab different:** most RAG tutorials use Wikipedia or PDF manuals as the
retrieval corpus, which is clean and doesn't fail in interesting ways. This lab uses a real
enterprise data source (Confluence) with a deliberately realistic failure: a stale, never-archived
runbook page that crowds out its current replacement. Neither of Phoenix's LLM-judge evaluators
catches this on their own — only a deterministic cross-check does, which is the actual point of
the lab.

It also runs **fully locally by default** (Ollama on Apple Silicon), including the eval judge —
zero cloud API keys required for the default path, so it doubles as workshop material trainees
can run entirely on their own MacBooks (see "Known limitations" below for the real tradeoff
that comes with a local judge).

## Why this lab exists

A real customer once asked a real company's support chatbot about a bereavement discount. The
bot said he could apply for it after he flew, within 90 days of the fare. He booked, he flew, he
filed the claim. The company said no.

Here is the part that matters: the bot's own answer linked to the policy page it was quoting.
That page said the opposite -- apply before you travel, not after. The bot's answer and its own
source contradicted each other, in the same reply, on the same site.

The company argued the bot was a separate legal entity, answerable for its own words. A tribunal
disagreed and ruled for the customer.

This was not a hallucination. The bot did not invent a policy. It retrieved real text and served
the wrong version of it, with full confidence. An old policy page and its current replacement sat
side by side in the same knowledge base, and the bot picked the old one.

This lab reproduces that exact failure -- an old page that crowds out its current replacement --
on a dummy Confluence wiki, so you can catch it, trace it, and fix it yourself.

## Architecture

```
Confluence (dummy "Meridian Analytics" wiki)
        |  ingest + chunk
        v
   Vector store  --(naive retrieval: no freshness awareness)-->  stale page retrieved
        |
        v
   LLM generates answer  --(faithful to the stale page)-->  confidently wrong answer
        |
        v
   Phoenix trace  (see exactly which page was retrieved, and what the LLM said)
        |
        v
   Phoenix evals: DocumentRelevanceEvaluator + FaithfulnessEvaluator both look fine
                  (the stale page IS topically relevant; the answer IS faithful to it)
                  --> only a deterministic "is this page deprecated?" check catches the failure
```

## Prerequisites

- **Local mode (default, zero API keys):** [Ollama](https://ollama.com) installed and running,
  with `qwen2.5`, `nomic-embed-text`, and `gemma2:9b` pulled (the last one is the judge
  model — deliberately different from the generation model, see "Known limitations"). A free
  Atlassian Cloud site with a Confluence space populated with the dummy Meridian Analytics pages
  (see the companion README/`data/confluence_seed_pages.md`), plus a Confluence API token.
- **Cloud mode (alternate path, e.g. for Colab):** an `OPENAI_API_KEY` covers both generation and
  the eval judge in this mode. Colab cannot run a persistent Ollama daemon, so set
  `LOCAL_MODE = False` before running there.

## Known limitations (stated up front, not hidden)

- **In `LOCAL_MODE`, the eval judge went through its own reliability check before being picked.**
  The first candidate, `mistral-nemo:12b`, was measurably inconsistent: over 10 repeated trials
  per failure case, `DocumentRelevanceEvaluator` was always consistent, but `FaithfulnessEvaluator`
  was not — solid on the database-failover case, but 8 "faithful" vs. 2 "unfaithful" on the
  password-reset case, identical input every time. `gemma2:9b` was tested the same way and came
  back fully consistent (10/10) on both cases, so it's now the default local judge. This history
  is left visible rather than smoothed over — a judge that looks consistent in one test still
  isn't ground truth, which is exactly why the lab's real conclusion rests on a deterministic
  check instead (see below).
- The judge model is deliberately **different** from the generation model (`qwen2.5`) to avoid
  "self-preference bias" — a model judging its own answer is one of the specific reliability
  problems called out below.
- LLM-as-judge evaluators carry real, documented reliability problems (a March 2026 RAND
  Corporation study found no LLM judge is uniformly reliable across benchmarks — position bias,
  verbosity bias, and self-preference bias are all measurable). This lab does not treat the
  Phoenix eval score as ground truth; it cross-checks it against a deterministic, non-LLM signal
  that doesn't depend on any model's mood.


## 1. Setup & install

In [2]:
import importlib.util

# `uv sync` (see README) already installs every dependency from pyproject.toml straight into
# this kernel's environment, and that environment has no pip module by design -- so this cell
# has nothing to do locally. It only does real work on Colab or any other plain-pip kernel.
if importlib.util.find_spec("pip") is not None:
    get_ipython().run_line_magic(
        "pip",
        "install -q llama-index llama-index-llms-ollama llama-index-embeddings-ollama "
        "llama-index-llms-openai llama-index-embeddings-openai llama-index-readers-confluence "
        "arize-phoenix arize-phoenix-evals openinference-instrumentation-llama-index litellm",
    )
else:
    print("No pip in this kernel -- expected with `uv sync` + `uv run jupyter notebook`. "
          "Packages already installed from pyproject.toml. Skipping.")


No pip in this kernel -- expected with `uv sync` + `uv run jupyter notebook`. Packages already installed from pyproject.toml. Skipping.


In [3]:
from dotenv import load_dotenv

load_dotenv()  # picks up a local .env if present; falls back to prompts below otherwise

import getpass
import os

import pandas as pd

import phoenix as px
from phoenix.client import Client
from phoenix.evals import LLM
from phoenix.evals.metrics import DocumentRelevanceEvaluator, FaithfulnessEvaluator
from openinference.instrumentation.llama_index import LlamaIndexInstrumentor

from llama_index.core import Settings, VectorStoreIndex
from llama_index.readers.confluence import ConfluenceReader


## 2. Configuration

Set `LOCAL_MODE = True` to run generation + embeddings on Ollama (default, zero API keys for this
part). Set `LOCAL_MODE = False` to run the same pipeline on OpenAI instead — useful for Colab,
where a persistent Ollama daemon isn't available.

**The eval judge follows the same flag.** In local mode the judge is `gemma2:9b` over Ollama;
in cloud mode it is `gpt-4o-mini` -- see "Known limitations" above for why the judge model is
always different from the generation model.


In [4]:
LOCAL_MODE = True  # False for cloud/Colab mode

OLLAMA_LLM_MODEL = "qwen2.5"
OLLAMA_EMBED_MODEL = "nomic-embed-text"
# Deliberately a DIFFERENT model from OLLAMA_LLM_MODEL -- using the same model to both generate
# and judge its own answer is "self-preference bias," one of the specific judge-reliability
# problems this notebook already flags. gemma2:9b replaced mistral-nemo:12b as the default judge
# after a repeated-trial reliability comparison found it more consistent (see "Known limitations"
# above). Pull it ahead of a live session with `ollama pull gemma2:9b`.
OLLAMA_JUDGE_MODEL = "gemma2:9b"
CLOUD_LLM_MODEL = "gpt-4o-mini"
CLOUD_EMBED_MODEL = "text-embedding-3-small"

if LOCAL_MODE:
    from llama_index.llms.ollama import Ollama
    from llama_index.embeddings.ollama import OllamaEmbedding

    Settings.llm = Ollama(model=OLLAMA_LLM_MODEL, request_timeout=120.0)
    Settings.embed_model = OllamaEmbedding(model_name=OLLAMA_EMBED_MODEL)
else:
    # Cloud mode needs an OpenAI key for generation AND judging -- local mode needs neither.
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

    from llama_index.llms.openai import OpenAI as LlamaOpenAI
    from llama_index.embeddings.openai import OpenAIEmbedding

    Settings.llm = LlamaOpenAI(model=CLOUD_LLM_MODEL)
    Settings.embed_model = OpenAIEmbedding(model=CLOUD_EMBED_MODEL)

CONFLUENCE_URL = os.environ.get("CONFLUENCE_URL") or input("Confluence site URL (e.g. https://your-site.atlassian.net/wiki): ")
CONFLUENCE_USERNAME = os.environ.get("CONFLUENCE_USERNAME") or input("Confluence account email: ")
CONFLUENCE_API_TOKEN = os.environ.get("CONFLUENCE_API_TOKEN") or getpass.getpass("Confluence API token: ")
CONFLUENCE_SPACE_KEY = os.environ.get("CONFLUENCE_SPACE_KEY", "ENG")


In [5]:
# Local-mode sanity check: fail with a friendly message instead of a stack trace deep in LlamaIndex.
if LOCAL_MODE:
    import requests

    try:
        requests.get("http://localhost:11434/api/tags", timeout=3)
    except requests.exceptions.ConnectionError as exc:
        raise RuntimeError(
            "Could not reach the Ollama daemon at localhost:11434. "
            "Start it with `ollama serve` (or open the Ollama app), and make sure "
            f"`ollama pull {OLLAMA_LLM_MODEL}` and `ollama pull {OLLAMA_EMBED_MODEL}` have been run."
        ) from exc
    print("Ollama daemon reachable.")


Ollama daemon reachable.


## 3. Phoenix instrumentation

`LlamaIndexInstrumentor` auto-instruments at the framework level via OpenInference, so tracing
works transparently regardless of whether LlamaIndex is calling Ollama or OpenAI underneath —
nothing below this cell needs to change based on `LOCAL_MODE`.


In [7]:
from phoenix.otel import register

session = px.launch_app()
tracer_provider = register(project_name="default", auto_instrument=False)
LlamaIndexInstrumentor().instrument(tracer_provider=tracer_provider)
tracer = tracer_provider.get_tracer("rag-lab-manual-spans")
print(f"Phoenix UI: {session.url}")


Existing running Phoenix instance detected! Shutting it down and starting a new instance...
/Users/rajeshr/projects/claude-code-projects/ai-agent-observability-phoenix/.venv/lib/python3.11/site-packages/pydantic/json_schema.py:2463: PydanticJsonSchemaWarning: Default value <phoenix.db.types.db_helper_types.Undefined object at 0x11f6598d0> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
boto3 is installed but aioboto3 is not. To use AWS Bedrock models in Playground, install aioboto3: pip install aioboto3
Overriding of current TracerProvider is not allowed
Attempting to instrument while already instrumented


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: default
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Phoenix UI: http://localhost:6006/


In [8]:
def run_traced_query(query_engine, query_str, span_name):
    """Wrap a query in a manually-started OTel span so we get back a trace_id we can use to
    unambiguously find THIS call's spans in Phoenix afterward -- much more reliable than
    assuming "the most recent span" is the right one, which breaks the moment cells get
    re-run out of order."""
    with tracer.start_as_current_span(span_name) as span:
        response = query_engine.query(query_str)
        ctx = span.get_span_context()
        wrapper_trace_id = format(ctx.trace_id, "032x")
        wrapper_span_id = format(ctx.span_id, "016x")
    return response, wrapper_trace_id, wrapper_span_id


def find_retriever_span(client, trace_id):
    """A single retrieval actually creates two RETRIEVER-kind spans (an outer `.retrieve`
    and an inner `._retrieve`) -- only the outer one carries the `retrieval.documents.*`
    attributes we need, so filter for that rather than assuming there's exactly one
    RETRIEVER span per query."""
    candidates = client.spans.get_spans(
        project_identifier="default", trace_ids=[trace_id], span_kind="RETRIEVER"
    )
    for s in candidates:
        if any(k.startswith("retrieval.documents") for k in s.get("attributes", {})):
            return s
    raise ValueError(f"No retriever span with document attributes found for trace {trace_id}")


## 4. Confluence ingestion

This assumes a Confluence space `ENG` populated with the ~19 dummy "Meridian Analytics" pages,
including the three failover-runbook pages that make up the deliberate failure scenario:

- "Database Failover Runbook (Old)" — the trap: long, keyword-dense, never archived despite being
  superseded.
- "Database Failover Runbook — Automated (Current)" — the correct page: short, sparse in
  matching tokens.
- "Incident Response Overview" — reinforces the trap via a stale internal link.

**Two real-world wrinkles surfaced when connecting to a live Cloud instance, both handled below:**
1. `ConfluenceReader`'s `api_token=` parameter authenticates as a Bearer token, which is for
   Confluence **Server/Data Center** Personal Access Tokens. Confluence **Cloud** needs Basic Auth
   instead — pass the API token via `password=`, with `user_name=` as your account email.
2. Hand-authoring pages by pasting Markdown produced literal `#`/`##` prefixes and stray marker
   emoji baked into the page titles (an artifact of copy-pasting, not a code bug). Titles are
   normalized below, and the result is filtered down to the known set of 19 pages — this also
   conveniently drops Confluence's own auto-created homepage and default template pages, which
   aren't part of the corpus.


In [9]:
import re

reader = ConfluenceReader(base_url=CONFLUENCE_URL, user_name=CONFLUENCE_USERNAME, password=CONFLUENCE_API_TOKEN)
raw_docs = reader.load_data(space_key=CONFLUENCE_SPACE_KEY, max_num_results=50)

def normalize_title(raw_title):
    """Strip leading Markdown heading markers and a trailing marker emoji from hand-pasted titles."""
    title = re.sub(r"^#+\s*", "", raw_title or "")
    title = re.sub(r"\s*\U0001F3AF\s*$", "", title)
    return title.strip()

EXPECTED_TITLES = {
    "Database Failover Runbook (Old)",
    "Database Failover Runbook — Automated (Current)",
    "Incident Response Overview",
    "API Rate Limiting Runbook",
    "On-Call Escalation Policy",
    "Deployment Rollback Procedure",
    "Welcome to Meridian Engineering",
    "Engineering Team Directory",
    "Local Dev Environment Setup",
    "Meridian Analytics API v2 Overview",
    "Custom Dashboard Builder Guide",
    "Data Export Formats Supported",
    "SSO Integration Guide (Okta)",
    "SOC 2 Compliance Overview",
    "Data Retention Policy",
    "Password Reset Policy (Legacy — Pre-SSO)",
    "Password Reset Policy (Current — SSO Managed)",
    "Common Customer FAQ — Billing",
    "Common Customer FAQ — Onboarding",
}

# Manual deprecation metadata (more reliable than Confluence label/CQL extraction for this demo).
DEPRECATED_TITLES = {"Database Failover Runbook (Old)", "Password Reset Policy (Legacy — Pre-SSO)"}

docs = []
for d in raw_docs:
    title = normalize_title(d.metadata.get("title", ""))
    if title not in EXPECTED_TITLES:
        continue  # drop the space homepage and Atlassian's default template pages
    d.metadata["title"] = title
    d.metadata["status"] = "deprecated" if title in DEPRECATED_TITLES else "current"
    docs.append(d)

missing = EXPECTED_TITLES - {d.metadata["title"] for d in docs}
if missing:
    print("WARNING - expected pages not found in the space:", missing)

sanity = pd.DataFrame(
    [{"title": d.metadata["title"], "length": len(d.text), "status": d.metadata["status"]} for d in docs]
)
sanity.sort_values("length", ascending=False)


,title,length,status
0,Database Failover Runbook (Old),3271,deprecated
18,Password Reset Policy (Legacy — Pre-SSO),1884,deprecated
3,Incident Response Overview,766,current
1,Database Failover Runbook — Automated (Current),728,current
4,API Rate Limiting Runbook,616,current
7,Welcome to Meridian Engineering,547,current
6,Deployment Rollback Procedure,497,current
12,Common Customer FAQ — Onboarding,431,current
5,On-Call Escalation Policy,421,current
2,Engineering Team Directory,414,current


## 5. Naive index construction (the "broken" baseline)

This is deliberately naive: default chunking, `similarity_top_k=1`, and **no awareness of the
`status` metadata field** — the index has no way to know "deprecated" means "don'''t retrieve
this." (Empirically, `top_k=1` is what actually reproduces the failure end-to-end here — at
`top_k=2` the model sees both the old and current runbook together and correctly synthesizes the
right answer anyway, which is a good reminder that RAG failures can be sensitive to a config value
as small as retrieval count.)


In [10]:
index = VectorStoreIndex.from_documents(docs)
naive_query_engine = index.as_query_engine(similarity_top_k=1)


## 6. Trigger the failure

Expected outcome: the stale "(Old)" runbook page outranks the current one in retrieval, and the
generated answer confidently describes the deprecated manual procedure as if it were current
guidance. (If your "(Old)" runbook page still contains an in-body deprecation banner, the LLM may
read that banner directly and self-correct — remove the banner from the live page for the failure
to manifest all the way through to the generated answer, not just at the retrieval step.)


In [11]:
TRIGGER_QUERY = "What is the current procedure for failing over our primary database during an incident?"

response, naive_trace_id, naive_wrapper_span_id = run_traced_query(
    naive_query_engine, TRIGGER_QUERY, "exercise1_naive_trigger"
)
print("ANSWER:\n", response.response)
print("\nRETRIEVED SOURCES:")
for node in response.source_nodes:
    print(f"  - {node.metadata.get('title')}  (score={node.score:.3f})")
print("\ntrace_id:", naive_trace_id)


ANSWER:
 The current documented procedure for failing over the primary database during an incident involves several steps:

1. Confirm SSH access to all database cluster nodes.
2. Ensure the replication lag on all replicas is under 5 seconds.
3. Notify the incident channel about the planned failover.
4. Execute these manual steps:
   a. SSH into the unresponsive primary database node and verify its unresponsiveness using `pg_isready -h localhost`.
   b. SSH into an intended replica, confirm its replication lag with `SELECT now() - pg_last_xact_replay_timestamp();`, then promote it to the primary role by running `pg_ctlcluster 14 main promote`.
   c. Verify the promoted node is accepting writes using `psql -c "SELECT pg_is_in_recovery();"`; it should return `f`.
   d. Update the internal DNS record for `db-primary.internal` with the new primary's IP address.
   e. If necessary, update the application’s `DATABASE_URL` secret by temporarily hardcoding the new primary’s IP in `/etc/hosts`.

## 7. Inspect the trace in Phoenix

Open the Phoenix UI link printed in section 3 in your browser. Find the trace named
`exercise1_naive_trigger` — that's the one this query just created (the code cell below also
prints its exact `naive_trace_id`, so you can confirm you're looking at the right one if your
Phoenix UI's trace list supports searching by ID).

Click into it. You should see three levels: the `exercise1_naive_trigger` root span (the one we
created manually as a wrapper), a `RETRIEVER` span underneath it, and an `LLM` span for the
generation step. Click the `RETRIEVER` span and expand its attributes — you'll find the exact
page title and text we just printed above, as `retrieval.documents.0.document.content`. This is
the actual retrieval Phoenix captured, not a re-derived summary — and it's the same span we're
about to attach eval results to below.


In [12]:
client = Client()
spans_df = client.spans.get_spans_dataframe().sort_values("start_time")
spans_df[["name", "span_kind", "status_code"]].tail(10)

,name,span_kind,status_code
context.span_id,,,
5f8b8622aae3f7d1,OllamaEmbedding.get_query_embedding,EMBEDDING,OK
72d1a74a0c4f78eb,OllamaEmbedding._get_query_embedding,EMBEDDING,OK
f5673b690ae0c837,CompactAndRefine.synthesize,CHAIN,OK
f4a0f67c48158e10,CompactAndRefine.get_response,CHAIN,OK
9dba5c7d80c4b7bc,TokenTextSplitter.split_text,CHAIN,OK
697c87bb79863dbe,CompactAndRefine.get_response,CHAIN,OK
9057cb9aac59c9b2,TokenTextSplitter.split_text,CHAIN,OK
9539efe5fc123ce4,DefaultRefineProgram.__call__,CHAIN,OK
2c33e7f55da8ba98,Ollama.predict,LLM,OK


In [13]:
# Unlike .tail(10) above (which just shows "whatever's recent"), this reliably finds THIS
# query's retriever span via its trace_id -- it stays correct even if cells get re-run out
# of order.
naive_retriever_span = find_retriever_span(client, naive_trace_id)
naive_retriever_span_id = naive_retriever_span["context"]["span_id"]
print("Retriever span for this query:", naive_retriever_span_id)


Retriever span for this query: c554fb3d8060eac2


## 8. Evals: relevance, faithfulness, and a deterministic cross-check

The eval judge follows `LOCAL_MODE` — a local Ollama model (`OLLAMA_JUDGE_MODEL`, deliberately
different from the generation model) by default, or `gpt-4o-mini` in cloud mode (see "Known
limitations" above for the reliability tradeoff that comes with a local judge).

In [14]:
judge = (
    LLM(provider="litellm", model=f"ollama/{OLLAMA_JUDGE_MODEL}")
    if LOCAL_MODE
    else LLM(provider="openai", model=CLOUD_LLM_MODEL)
)

relevance_evaluator = DocumentRelevanceEvaluator(llm=judge)
faithfulness_evaluator = FaithfulnessEvaluator(llm=judge)

retrieved_text = response.source_nodes[0].node.get_content()

relevance_scores = relevance_evaluator.evaluate({"input": TRIGGER_QUERY, "document_text": retrieved_text})
faithfulness_scores = faithfulness_evaluator.evaluate(
    {"input": TRIGGER_QUERY, "output": response.response, "context": retrieved_text}
)

print("DocumentRelevanceEvaluator:", [(s.label, s.score) for s in relevance_scores])
print("FaithfulnessEvaluator:    ", [(s.label, s.score) for s in faithfulness_scores])

# Deterministic, non-LLM cross-check: did retrieval surface a page we know is deprecated?
def deterministic_deprecation_check(source_nodes):
    retrieved_titles = {n.metadata.get("title") for n in source_nodes}
    hit = retrieved_titles & DEPRECATED_TITLES
    return {"passed": not hit, "deprecated_pages_retrieved": sorted(hit)}

print("Deterministic check:       ", deterministic_deprecation_check(response.source_nodes))


def log_retrieval_annotations_to_phoenix(
    client, *, wrapper_span_id, retriever_span_id, source_nodes,
    relevance_scores, faithfulness_scores, deprecated_titles,
):
    """Writes eval + deterministic-check results into Phoenix as annotations, so they're
    visible on the trace in the UI -- not just printed to stdout and lost when the kernel
    restarts. sync=True (the API defaults to False) is a deliberate lab-only tradeoff: we want
    the annotation to be there the instant you refresh the browser, not queued asynchronously.
    In a real high-throughput pipeline you'd normally leave sync=False.
    Calls are idempotent (keyed by span_id + annotation_name), so re-running this cell while
    you experiment won't create duplicate annotations."""
    for i, node in enumerate(source_nodes):
        client.spans.add_document_annotation(
            span_id=retriever_span_id, document_position=i,
            annotation_name="document_relevance", annotator_kind="LLM",
            label=relevance_scores[i].label, score=relevance_scores[i].score,
            explanation=relevance_scores[i].explanation, sync=True,
        )
        is_deprecated = node.metadata.get("title") in deprecated_titles
        client.spans.add_document_annotation(
            span_id=retriever_span_id, document_position=i,
            annotation_name="deprecation_check", annotator_kind="CODE",
            label="deprecated" if is_deprecated else "current",
            score=0.0 if is_deprecated else 1.0, sync=True,
        )
    client.spans.add_span_annotation(
        span_id=wrapper_span_id, annotation_name="faithfulness",
        annotator_kind="LLM", label=faithfulness_scores[0].label,
        score=faithfulness_scores[0].score,
        explanation=faithfulness_scores[0].explanation, sync=True,
    )


log_retrieval_annotations_to_phoenix(
    client, wrapper_span_id=naive_wrapper_span_id, retriever_span_id=naive_retriever_span_id,
    source_nodes=response.source_nodes, relevance_scores=relevance_scores,
    faithfulness_scores=faithfulness_scores, deprecated_titles=DEPRECATED_TITLES,
)
print("Annotations logged to Phoenix.")


DocumentRelevanceEvaluator: [('relevant', 1.0)]
FaithfulnessEvaluator:     [('faithful', 1.0)]
Deterministic check:        {'passed': False, 'deprecated_pages_retrieved': ['Database Failover Runbook (Old)']}
Annotations logged to Phoenix.


**A note on what you might see here:** in `LOCAL_MODE`, the judge is a small model
(`gemma2:9b`) running on your own machine, not a large cloud model. During judge selection we
compared it against `mistral-nemo:12b`, the earlier default, over 10 repeated trials per failure
case on identical input: `mistral-nemo:12b` split 8/2 on the password-reset case, while
`gemma2:9b` came back fully consistent (10/10) on both cases. A single run in this cell won't
reproduce that split, so don't expect to see disagreement here -- but a judge that was consistent
across 10 trials in one test still isn't proven reliable in general, and that's exactly why the
deterministic check below, not this judge, is what the lab's conclusion rests on.


Go check the Phoenix UI now. Refresh the trace you opened in section 7 — you should see a
`document_relevance` label and a `deprecation_check` label (flagged "deprecated") on the
retriever span's retrieved document, and a `faithfulness` label on the root span. This is the
exact same pass/fail split you saw printed above, but now it's a durable, queryable, UI-visible
record instead of a printed line that disappears when the kernel restarts.


## 9. Diagnosis

**Confirmed empirically against the live Confluence content (`qwen2.5` + `nomic-embed-text`
locally, `gpt-4o-mini` as judge):**

| Check | Result |
|---|---|
| `DocumentRelevanceEvaluator` | **relevant**, score 1.0 — "the document ... directly addresses the current procedure ... as requested" |
| `FaithfulnessEvaluator` | **faithful**, score 1.0 — "the response correctly summarizes ... the detailed manual failover procedure ... No incorrect information is presented" |
| Deterministic deprecation check | **failed** — the retrieved page (`Database Failover Runbook (Old)`) is in the known-deprecated set |

Both LLM judges are right, on their own terms: the retrieved page genuinely is topically relevant,
and the generated answer genuinely is faithful to what got retrieved. Neither evaluator has any
concept of "current vs. deprecated" — that is not what either metric measures. Only the
deterministic check, which knows about the `status` metadata field, catches that the on-call
engineer just received a confident, well-cited, fully-grounded answer that walks through a
Postgres failover procedure the infrastructure hasn'''t used since before it was migrated to RDS
Multi-AZ.

**That gap is the core insight of this notebook: tracing plus LLM-judge evals were not enough on
their own to catch this failure. A domain-specific, deterministic check caught something both
probabilistic judges missed by construction, not by bad luck.**


## 10. Exercise 1 — fix it yourself

You just saw both LLM judges confidently pass a wrong answer. Now fix the actual retrieval
config.

**Your task:** the `docs` you loaded in section 4 already carry a `status` metadata field
(`"current"` or `"deprecated"`) — that's exactly the signal the naive index in section 5 ignored.
Build a new query engine, `fixed_query_engine`, from the same `index`, that **excludes any node
where `status == "deprecated"`**, then re-run `TRIGGER_QUERY` against it.

Hints:
- LlamaIndex query engines accept a `filters=` argument built from
  `llama_index.core.vector_stores.MetadataFilters` and `ExactMatchFilter`.
- You don't need to rebuild the index — the metadata is already attached to every node in it.

Try it in the cell below before looking at the solution further down.


In [15]:
# YOUR CODE HERE
#
# from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter
#
# fixed_query_engine = index.as_query_engine(
#     similarity_top_k=...,
#     filters=...,
# )
#
# fixed_response = fixed_query_engine.query(TRIGGER_QUERY)
# print(fixed_response.response)


### Solution (Exercise 1)

Only look once you've tried it yourself.


In [16]:
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter

fixed_query_engine = index.as_query_engine(
    similarity_top_k=2,
    filters=MetadataFilters(filters=[ExactMatchFilter(key="status", value="current")]),
)

fixed_response = fixed_query_engine.query(TRIGGER_QUERY)
print("FIXED ANSWER:\n", fixed_response.response)
print("\nFIXED RETRIEVED SOURCES:")
for node in fixed_response.source_nodes:
    print(f"  - {node.metadata.get('title')}  (score={node.score:.3f})")


FIXED ANSWER:
 The current procedure for failing over the primary database during an incident involves a fully automated process via RDS Multi-AZ failover, orchestrated by the Terraform module `infra/db-failover`. In normal circumstances, if the primary instance becomes unresponsive, AWS RDS automatically promotes the standby replica and updates the DNS CNAME, requiring no manual intervention.

However, in rare cases where automation fails, manual intervention is necessary. This scenario is detailed in Appendix B of the platform team's internal incident wiki. If you are paged for a database incident and the automated failover does not complete within 5 minutes, you should escalate to the Platform Engineering on-call lead instead of attempting a manual procedure.

FIXED RETRIEVED SOURCES:
  - Database Failover Runbook — Automated (Current)  (score=0.725)
  - Incident Response Overview  (score=0.724)


## 11. Before/after comparison


In [17]:
comparison = pd.DataFrame([
    {
        "variant": "naive (before)",
        "retrieved_titles": [n.metadata.get("title") for n in response.source_nodes],
        "deterministic_check_passed": deterministic_deprecation_check(response.source_nodes)["passed"],
    },
    {
        "variant": "fixed (after)",
        "retrieved_titles": [n.metadata.get("title") for n in fixed_response.source_nodes],
        "deterministic_check_passed": deterministic_deprecation_check(fixed_response.source_nodes)["passed"],
    },
])
comparison


,variant,retrieved_titles,deterministic_check_passed
0,naive (before),[Database Failover Runbook (Old)],False
1,fixed (after),[Database Failover Runbook — Automated (Curren...,True


## 12. Exercise 2 — diagnose and fix on your own

Now run the full cycle yourself, on a different failure case, using the same `index` you already
built (it contains all 19 pages, so no new ingestion is needed).

**The scenario:** the space also has a second stale/duplicate pair — "Password Reset Policy
(Legacy — Pre-SSO)" (deprecated) vs. "Password Reset Policy (Current — SSO Managed)" (current).

**New trigger query:** `"Can our support team manually reset a customer password?"`

Work through the same three steps as Exercise 1 — try each one yourself before checking the
solution:

1. Build a naive query engine (`similarity_top_k=1`) from `index` and run the query above. Which
   page gets retrieved, and is the answer actually correct for an SSO-managed customer today?
2. Run `relevance_evaluator` and `faithfulness_evaluator` (already instantiated in section 8)
   against the retrieved page/answer, plus a deterministic deprecation check like the one from
   section 8. Do the judges agree with the deterministic check?
3. Build a `status`-filtered query engine, same pattern as Exercise 1's solution, and re-run the
   query. Does the answer change?


In [ ]:
# Step 1: naive retrieval — YOUR CODE HERE
#
# ex2_query = "Can our support team manually reset a customer password?"
# ex2_naive_engine = index.as_query_engine(similarity_top_k=...)
# ex2_naive_response = ex2_naive_engine.query(ex2_query)
# print(ex2_naive_response.response)


### Solution (Exercise 2, step 1)

Only look once you've tried it yourself.


In [18]:
ex2_query = "Can our support team manually reset a customer password?"
ex2_naive_engine = index.as_query_engine(similarity_top_k=1)
ex2_naive_response, ex2_trace_id, ex2_wrapper_span_id = run_traced_query(
    ex2_naive_engine, ex2_query, "exercise2_naive_trigger"
)

print("ANSWER:\n", ex2_naive_response.response)
print("\nRETRIEVED SOURCES:")
for node in ex2_naive_response.source_nodes:
    print(f"  - {node.metadata.get('title')}  (status={node.metadata.get('status')}, score={node.score:.3f})")


ANSWER:
 Yes, our support team can manually reset a customer's password in rare cases where they cannot access their registered email at all. To do this, they need to verify the user's identity using the standard identity verification checklist and then manually reset the account to a temporary password from the admin console.

RETRIEVED SOURCES:
  - Password Reset Policy (Legacy — Pre-SSO)  (status=deprecated, score=0.719)


In [ ]:
# Step 2: evals + deterministic check — YOUR CODE HERE
#
# ex2_retrieved_text = ex2_naive_response.source_nodes[0].node.get_content()
# ex2_relevance = relevance_evaluator.evaluate({...})
# ex2_faithfulness = faithfulness_evaluator.evaluate({...})
# print(deterministic_deprecation_check(ex2_naive_response.source_nodes))


### Solution (Exercise 2, step 2)

Only look once you've tried it yourself.


In [ ]:
ex2_retrieved_text = ex2_naive_response.source_nodes[0].node.get_content()

ex2_relevance = relevance_evaluator.evaluate({"input": ex2_query, "document_text": ex2_retrieved_text})
ex2_faithfulness = faithfulness_evaluator.evaluate(
    {"input": ex2_query, "output": ex2_naive_response.response, "context": ex2_retrieved_text}
)

print("DocumentRelevanceEvaluator:", [(s.label, s.score) for s in ex2_relevance])
print("FaithfulnessEvaluator:     ", [(s.label, s.score) for s in ex2_faithfulness])
print("Deterministic check:       ", deterministic_deprecation_check(ex2_naive_response.source_nodes))


ex2_retriever_span = find_retriever_span(client, ex2_trace_id)
ex2_retriever_span_id = ex2_retriever_span["context"]["span_id"]

log_retrieval_annotations_to_phoenix(
    client, wrapper_span_id=ex2_wrapper_span_id, retriever_span_id=ex2_retriever_span_id,
    source_nodes=ex2_naive_response.source_nodes, relevance_scores=ex2_relevance,
    faithfulness_scores=ex2_faithfulness, deprecated_titles=DEPRECATED_TITLES,
)
print("Annotations logged to Phoenix.")


(Same as Exercise 1 — refresh Phoenix to see these annotations on this trace too.)


In [ ]:
# Step 3: fix it — YOUR CODE HERE
#
# ex2_fixed_engine = index.as_query_engine(similarity_top_k=..., filters=...)
# ex2_fixed_response = ex2_fixed_engine.query(ex2_query)
# print(ex2_fixed_response.response)


### Solution (Exercise 2, step 3)

Only look once you've tried it yourself.


In [ ]:
ex2_fixed_engine = index.as_query_engine(
    similarity_top_k=2,
    filters=MetadataFilters(filters=[ExactMatchFilter(key="status", value="current")]),
)
ex2_fixed_response = ex2_fixed_engine.query(ex2_query)

print("FIXED ANSWER:\n", ex2_fixed_response.response)
print("\nFIXED RETRIEVED SOURCES:")
for node in ex2_fixed_response.source_nodes:
    print(f"  - {node.metadata.get('title')}  (score={node.score:.3f})")


**Confirmed empirically:** the legacy page outranks the current one at `top_k=1`, the generated
answer confidently says support *can* manually reset an SSO customer's password (wrong — per the
current policy, support cannot touch SSO-managed passwords at all, and that's a security-relevant
mistake, not just a stale-facts one). Both LLM judges again score the retrieved page **relevant**
and the answer **faithful** — same blind spot as Exercise 1, on completely different content. The
deterministic check catches it the same way. Two different scenarios, same failure mode, same fix.


## 13. (Stretch goal — drop first if short on time) Dual-judge comparison

During judge selection (see "Known limitations"), the earlier local judge (`mistral-nemo:12b`)
disagreed with itself across repeated runs on identical input. Take that further here: run
`FaithfulnessEvaluator` with a cloud judge (`gpt-4o-mini`, needs an `OPENAI_API_KEY`) against the
same failure case and compare it to the local judge (`gemma2:9b`) used above. This ties directly
to the judge-reliability research cited in "Known limitations": if judges disagree with each
other -- and, as the earlier local judge showed, even with themselves -- that's direct evidence
no single LLM judge should be treated as ground truth.

**Cut this section first if behind schedule** — the notebook's core insight (section 9) stands on
its own without it.


**Also optional, bigger lift:** Phoenix's Datasets & Experiments feature
(`client.datasets.create_dataset(...)`, `client.experiments.run_experiment(dataset=..., task=..., evaluators=...)`)
is the idiomatic way to compare the naive vs. fixed query engine side-by-side *inside the Phoenix
UI*, rather than in the pandas table in section 11. It's real complexity, not a drop-in: `task`
needs the signature `task(example) -> Any`, and evaluators need to match a specific type shape.
**This is not required for this lab** — the pandas before/after table already does the comparison
job. If you want to try it as a take-home: build a small dataset (one row per exercise's trigger
query), a task that runs both `naive_query_engine.query` and `fixed_query_engine.query` and
returns both answers plus retrieved titles, and wrap `deterministic_deprecation_check` as a custom
evaluator function.


In [ ]:
# local_judge = LLM(provider="litellm", model=f"ollama/{OLLAMA_LLM_MODEL}")
# local_faithfulness_evaluator = FaithfulnessEvaluator(llm=local_judge)
#
# cloud_result = faithfulness_evaluator.evaluate(response=response.response, context=[n.text for n in response.source_nodes])
# local_result = local_faithfulness_evaluator.evaluate(response=response.response, context=[n.text for n in response.source_nodes])
# print("Cloud judge:", cloud_result)
# print("Local judge:", local_result)


## 14. Wrap-up

**Takeaways:**
- A retrieval failure can be topically relevant and answer-faithful at the same time — and still
  be operationally wrong. Neither property that Phoenix's LLM judges check for is the same as
  "current" or "not deprecated."
- Tracing tells you *what happened*; LLM-judge evals tell you a probabilistic guess at *whether it
  was good*; neither substitutes for a deterministic, domain-specific check when one is available.
- This pattern — stale-but-plausible content beating short-but-correct content in dense retrieval
  — generalizes past Confluence to any enterprise knowledge source with a document lifecycle
  (Jira, Salesforce knowledge articles, internal wikis in general). You saw it reproduce on two
  unrelated topics (database failover, password resets) with the exact same fix.

**Status:** this started as a personal/workshop artifact. Proposing it as a Phoenix repo PR is a
separate, later step gated on maintainer sign-off — Phoenix's CONTRIBUTING.md states they are not
currently accepting unsolicited contributions.
